## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [1]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np

np.random.seed(42)  # random seed for reproducability

In [2]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [ ]:
assert "Train" not in business_covariates, "training set already assigned!"

# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))

train_indices = indices[:500]
calibration_indices = indices[500:700]
eval_indices = indices[700:]

# WICHTIG: Kopie für benchmark_covariates BEVOR sortiert wird
# Diese Kopie behält die Original-Reihenfolge und Indices
business_covariates_unsorted = business_covariates.copy()

# assign 'Train' variable: set to 1 for train_indices, 0 otherwise
business_covariates["Train"] = 0
business_covariates.loc[train_indices, "Train"] = 1

# sort business_covariates so that rows with Train==1 come first
# Diese Sortierung ist für Stan HMM wichtig, aber benchmark_covariates nutzt die unsortierte Version

business_covariates = business_covariates.sort_values(
    by="Train", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :N_train


n_train = len(business_covariates["Train"])  # number of training samples

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")
print(
    f"\nIMPORTANT: business_covariates_unsorted keeps original order for benchmark models!"
)

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 200 samples
  Eval: 221 samples

IMPORTANT: business_covariates_unsorted keeps original order for benchmark models!


In [4]:
# initialize new lists

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")

[0] - Conversion for business_id: QkG3KUXwqZBW18A9k1xqCA
[100] - Conversion for business_id: yTTEhnUOirRjGjxkObQTVw
[200] - Conversion for business_id: OdViVhR2ayppzkN2WtIScw
[300] - Conversion for business_id: WiqmuzPxWGiWDPxdSuVCXw
[400] - Conversion for business_id: yH83tf58E9jrME5U4HGAMg
[500] - Conversion for business_id: -PJgh1XoQBMnnSgg6MhmMA
[600] - Conversion for business_id: P7j_K9baGxWPlInbjn0OOg
[700] - Conversion for business_id: jcw_vtqfZSTLP15DJmvfIA
[800] - Conversion for business_id: OH3baEaklANPe1farAKgRg
[900] - Conversion for business_id: 3dsvREiTlmGaaBjsBS4dwQ
Done ... validation checks passed!


In [5]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe (both sorted and unsorted versions)
business_covariates["Age"] = age
business_covariates_unsorted["Age"] = age

# R: mutate(l_age = log(Age), Checkin = Checkin/Age*28)
# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)
business_covariates_unsorted["Checkin"] = (
    business_covariates_unsorted["Checkin"] / business_covariates_unsorted["Age"] * 28
)

# add log of age to dataframe for later analysis (same as R: l_age)
business_covariates["logAge"] = np.log(business_covariates["Age"])
business_covariates_unsorted["logAge"] = np.log(business_covariates_unsorted["Age"])

# R: Covariates <- covariates_business %>%
#    select(density, Checkin, category, chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age)
# only get relevant covariates for analysis (must match R order and selection)
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,American,1,2.0,700.0,350.0,1248.0,2057
1,15,0.744788,Salad,1,1.0,250.0,100.0,1638.0,2782
2,10,3.438404,American,0,1.0,247.0,186.0,1431.0,2557
3,2,1.276453,Asian,0,1.0,260.0,148.0,1223.0,2391
4,6,0.317627,Pizza,1,1.0,62.0,22.0,1398.0,2292
...,...,...,...,...,...,...,...,...,...
916,1,2.700297,American,0,3.0,207.0,128.0,1892.0,1348
917,11,25.242424,Cafes,0,2.0,496.0,356.0,1488.0,132
918,2,2.508399,Cafes,1,1.0,206.0,112.0,1288.0,893
919,2,1.698153,Cafes,0,1.0,119.0,64.0,1303.0,2869


In [6]:
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding and Scaling  already performed on dataframe!"

# R: options(na.action="na.pass")
# R: cov_mat <- model.matrix(formula(paste("~",paste(names(Covariates),collapse = "+"),"-1")),
#                             data = Covariates)[,-8]

# The formula in R is: ~ density + Checkin + category + chain + Price.Level +
#                        Restaurant.Size + Number.of.Seats + ZRI + Age - 1
# model.matrix creates columns in this order:
# density, Checkin, categoryAmerican, categoryAsian, categoryCafes, categoryFast Food,
# categoryMexican, categoryOther, categoryPizza, categorySalad, categorySpeciality Food,
# chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age
# Then [,-8] removes column 8 which is "categoryOther"

# Convert category to factor (like R)
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# Create dummy variables - R's model.matrix with "-1" creates all categories (no baseline)
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

# R model.matrix order: numeric columns in original order, then categorical dummies alphabetically
# Original order: density, Checkin, category (becomes multiple), chain, Price.Level,
#                 Restaurant.Size, Number.of.Seats, ZRI, Age

# First two numeric columns before category
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# Combine in R's order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# R: [,-8] removes column 8 (1-indexed in R)
# This is column index 7 in Python (0-indexed)
# Based on the order above, column 8 is "categoryOther"
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (R column 8): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")
        # Still remove it to match R behavior
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (R column 8): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057


In [7]:
# R: preProc <- preProcess(cov_mat[1:500,], c("center","medianImpute"))
# Python equivalent: First fit on training data, then center, then impute
# Note: R's caret::preProcess with c("center", "medianImpute") first centers, then imputes with median

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler(with_std=False)  # only centering, no scaling!


X_train = relevant_covariates.iloc[:n_train].copy()

# R applies: preProcess(cov_mat[1:500,], c("center","medianImpute"))
# This means: calculate center from training data, then impute missing values with median
# In R's caret, the order in the vector matters - "center" is applied first to calculate statistics
# but "medianImpute" fills NAs before centering is applied in the transform step

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition (same as R: qr() function)
Q, R = np.linalg.qr(X_train_preprocessed)

# R: Q <- qr.Q(QR)*sqrt(N_train-1)
# R: R <- qr.R(QR)/sqrt(N_train-1)
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

In [ ]:
from helpers import comp_entropy

# aggregate review stats
# R column names: VAR, MEAN, ENTR, COUNT, ONE_STAR, TWO_STAR, THREE_STAR, FOUR_STAR, FIVE_STAR
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities (same as R)
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# WICHTIG: Verwende business_covariates_unsorted um die Original-Indices zu behalten!
# R: select(business_id, density, Checkin, category, chain, Price.Level,
#           Restaurant.Size, Number.of.Seats, ZRI, Distance.To.City.Centre, Age, is_open)
benchmark_covariates = business_covariates_unsorted[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# R: mutate(Closed = 1-is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# R: left_join(temp, by="business_id")
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# R: mutate(l_COUNT = log(COUNT), category = factor(category), Closed = factor(Closed))
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical (same as R factors)
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# R: Closed <- fct_recode(Closed, "Closed" = "1", "Open" = "0")
# Convert Closed to categorical with proper labels
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")
print(
    f"WICHTIG: benchmark_covariates nutzt unsortierte Reihenfolge, train_indices sind korrekt!"
)

benchmark_covariates

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Benchmark covariates prepared with shape: (921, 23)
WICHTIG: benchmark_covariates nutzt unsortierte Reihenfolge, train_indices sind korrekt!


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,VAR,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,l_COUNT
0,QkG3KUXwqZBW18A9k1xqCA,11,2.327662,American,1,2.0,700.0,350.0,1248.0,14035.314000,...,2.900901,2.648649,1.421063,37,0.432432,0.108108,0.081081,0.135135,0.243243,3.610918
1,5XejqzaFmtkZMstJS5Iy-w,17,0.593817,American,0,1.0,200.0,30.0,1557.0,3279.013976,...,2.417417,3.837838,1.243687,37,0.189189,0.027027,0.054054,0.216216,0.513514,3.610918
2,M3uV9Y3EDSpy9d4YwyNSAQ,1,0.919828,Asian,0,2.0,280.0,70.0,1291.0,16198.365529,...,1.584382,4.015152,1.314273,66,0.075758,0.075758,0.090909,0.272727,0.484848,4.189655
3,U1ZVgF-kfkvv_rcoe0RglQ,4,4.391468,Asian,0,1.0,90.0,35.0,1557.0,4235.448723,...,1.199102,4.313953,1.103060,172,0.046512,0.046512,0.069767,0.220930,0.616279,5.147494
4,irYAkavSVtIEyGOsHK-yPw,4,1.722513,American,0,2.0,250.0,120.0,1495.0,1711.281915,...,1.959025,3.604651,1.422873,43,0.162791,0.023256,0.186047,0.302326,0.325581,3.761200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,u--jf3lS_04kmQNO4liAsQ,4,8.952522,Salad,1,1.0,220.0,70.0,1710.0,8526.356346,...,2.217876,3.508475,1.496835,118,0.135593,0.186441,0.093220,0.203390,0.381356,4.770685
917,0kD9yJIB2cmvOh8-WbcO_w,7,20.575758,Asian,0,2.0,190.0,24.0,1710.0,7016.984698,...,2.007459,3.515152,1.535840,66,0.136364,0.121212,0.166667,0.242424,0.333333,4.189655
918,UwZUPI7VAQAZZ0Gv0LV8nw,2,2.947368,Pizza,0,2.0,120.0,84.0,1638.0,25621.739627,...,1.258621,3.517241,1.430414,29,0.068966,0.103448,0.241379,0.413793,0.172414,3.367296
919,AEYNihHmGIjmUciRFo3qwA,4,0.478215,Asian,0,1.0,182.0,52.0,1287.0,5558.840743,...,1.431721,4.026316,1.207750,38,0.000000,0.184211,0.131579,0.157895,0.526316,3.637586


In [9]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=train_indices,
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / "processed_data.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 921
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (921, 16)
- R matrix: (16, 16)
- X_test: empty

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 200
- Eval indices: 221

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...

